# Safehouse Operational Load vs Incident Risk

## 1) Problem framing
**Business question:** How does safehouse operational intensity relate to incident burden per resident?

- Explanatory goal: estimate how process/home-visit intensity and occupancy correlate with incident rate.
- Predictive goal: flag months where rolling incident load is elevated vs that house's own baseline (staffing watchlist).
- Success metrics: explanatory R2/MAE; predictive ROC-AUC, F1, recall under GroupKFold by safehouse.

Deployment candidate: `/api/reports/trends/safehouse-load-risk` and Operations watchlist card.


## Target definition (reframed)

**Previous issue:** Global 75th percentile on `incident_rate` often collapses to a single class when the monthly snapshot is homogeneous.

**Current definition:**
- **Primary:** `above_roll3_median` — rolling 3-month incident rate (incidents per average resident in the window) is **above that safehouse's expanding median** of the same quantity, using **prior months only** (`shift(1)` on the expanding median) so the benchmark does not include the current month.
- **Fallback:** If that target is still degenerate, `mom_ir_positive` — month-over-month increase in raw `incident_rate`.

**Regression anchor:** `log1p(incident_rate)` remains the continuous outcome for explanatory analysis.

Rolling and lag features capture operational momentum without requiring extra tables.


In [ ]:

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import f1_score, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'data_loader.py').exists():
        sys.path.insert(0, str(candidate))
        break
from data_loader import load_table

m = load_table('safehouse_monthly_metrics').sort_values(['safehouse_id', 'month_start']).reset_index(drop=True).copy()
m.loc[:, 'incident_rate'] = (m['incident_count'] / m['active_residents'].replace(0, np.nan)).fillna(0)
m.loc[:, 'proc_per_resident'] = (m['process_recording_count'] / m['active_residents'].replace(0, np.nan)).fillna(0)
m.loc[:, 'visit_per_resident'] = (m['home_visitation_count'] / m['active_residents'].replace(0, np.nan)).fillna(0)
m.loc[:, 'month_num'] = m['month_start'].dt.month

def enrich_group(sub: pd.DataFrame) -> pd.DataFrame:
    sub = sub.copy()
    for w in (3, 6):
        sub.loc[:, f'roll{w}_incident_sum'] = sub['incident_count'].rolling(w, min_periods=1).sum()
        sub.loc[:, f'roll{w}_process_sum'] = sub['process_recording_count'].rolling(w, min_periods=1).sum()
        sub.loc[:, f'roll{w}_visit_sum'] = sub['home_visitation_count'].rolling(w, min_periods=1).sum()
        sub.loc[:, f'roll{w}_res_mean'] = sub['active_residents'].rolling(w, min_periods=1).mean().replace(0, np.nan)
        sub.loc[:, f'roll{w}_rate'] = sub[f'roll{w}_incident_sum'] / sub[f'roll{w}_res_mean']
        sub.loc[:, f'roll{w}_rate'] = sub[f'roll{w}_rate'].fillna(0)
    sub.loc[:, 'lag1_incident_rate'] = sub['incident_rate'].shift(1).fillna(0)
    sub.loc[:, 'lag1_proc_per_res'] = sub['proc_per_resident'].shift(1).fillna(0)
    sub.loc[:, 'lag1_visit_per_res'] = sub['visit_per_resident'].shift(1).fillna(0)
    sub.loc[:, 'mom_incident_rate'] = sub['incident_rate'].diff().fillna(0)
    sub.loc[:, 'roll3_rate_lag1'] = sub['roll3_rate'].shift(1).fillna(0)
    sub.loc[:, 'roll6_rate_lag1'] = sub['roll6_rate'].shift(1).fillna(0)
    r3 = sub['roll3_rate']
    sub.loc[:, 'roll3_median_prior'] = r3.expanding().median().shift(1)
    sub.loc[:, 'above_roll3_median'] = ((r3 > sub['roll3_median_prior']) & sub['roll3_median_prior'].notna()).astype(int)
    return sub

parts = [enrich_group(sub) for _, sub in m.groupby('safehouse_id', sort=False)]
m = pd.concat(parts, ignore_index=True)

m.loc[:, 'target_log_rate'] = np.log1p(m['incident_rate'].clip(lower=0))

y_cls = m['above_roll3_median']
cls_name = 'above_roll3_median'
if y_cls.nunique() < 2:
    y_cls = (m['mom_incident_rate'] > 0).astype(int)
    cls_name = 'mom_ir_positive'

# Predictive/explanatory features: exclude same-window roll3_rate/roll6_rate and their sums — they
# trivially determine above_roll3_median. Use lagged rolling signals and contemporaneous ops counts only.
features = [
    'safehouse_id', 'active_residents', 'avg_education_progress', 'avg_health_score',
    'process_recording_count', 'home_visitation_count',
    'proc_per_resident', 'visit_per_resident', 'month_num',
    'roll3_rate_lag1', 'roll6_rate_lag1',
    'lag1_incident_rate', 'lag1_proc_per_res', 'lag1_visit_per_res', 'mom_incident_rate',
]
X = m[features].copy()
y_reg = m['target_log_rate']
groups = m['safehouse_id'].values

houses = m['safehouse_id'].unique()
rng = np.random.RandomState(42)
rng.shuffle(houses)
n_hold = max(1, int(0.2 * len(houses)))
hold_set = set(houses[:n_hold])
train_mask = ~m['safehouse_id'].isin(hold_set)
test_mask = m['safehouse_id'].isin(hold_set)

Xtr, Xte = X.loc[train_mask], X.loc[test_mask]
ytr_reg, yte_reg = y_reg.loc[train_mask], y_reg.loc[test_mask]
ytr_clf, yte_clf = y_cls.loc[train_mask], y_cls.loc[test_mask]

cat_cols = ['safehouse_id']
num_cols = [c for c in features if c not in cat_cols]
prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

lin = Pipeline([('prep', prep), ('model', LinearRegression())])
lin.fit(Xtr, ytr_reg)
pred_reg = lin.predict(Xte)
print('Explanatory R2:', round(r2_score(yte_reg, pred_reg), 3))
print('Explanatory MAE:', round(mean_absolute_error(yte_reg, pred_reg), 3))
print('Classification target:', cls_name, 'nunique=', y_cls.nunique())

n_splits = min(5, max(2, m['safehouse_id'].nunique()))
gkf = GroupKFold(n_splits=n_splits)

if y_cls.nunique() < 2:
    print('Predictive track skipped: classification target degenerate.')
else:
    baseline = Pipeline([('prep', prep), ('model', DummyClassifier(strategy='prior'))])
    rf = Pipeline([('prep', prep), ('model', RandomForestClassifier(
        n_estimators=250, random_state=42, min_samples_leaf=3, class_weight='balanced'))])
    baseline.fit(Xtr, ytr_clf)
    rf.fit(Xtr, ytr_clf)
    if yte_clf.nunique() < 2:
        print('Holdout has single class; reporting GroupKFold CV only.')
    else:
        base_proba = baseline.predict_proba(Xte)[:, 1]
        rf_proba = rf.predict_proba(Xte)[:, 1]
        rf_pred = (rf_proba >= 0.5).astype(int)
        print('Baseline AUC:', round(roc_auc_score(yte_clf, base_proba), 3))
        print('RF AUC:', round(roc_auc_score(yte_clf, rf_proba), 3))
        print('RF F1:', round(f1_score(yte_clf, rf_pred, zero_division=0), 3))

    cv = cross_validate(rf, X, y_cls, cv=gkf, scoring=['roc_auc', 'f1'], groups=groups, n_jobs=-1)
    print('GroupKFold CV AUC mean/std:', round(float(cv['test_roc_auc'].mean()), 3), round(float(cv['test_roc_auc'].std()), 3))
    print('GroupKFold CV F1 mean/std:', round(float(cv['test_f1'].mean()), 3), round(float(cv['test_f1'].std()), 3))

    if yte_clf.nunique() >= 2:
        perm = permutation_importance(rf, Xte, yte_clf, n_repeats=8, random_state=42, scoring='roc_auc')
        imp = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).head(10)
        print('\nTop predictive drivers:')
        print(imp.round(4).to_string())

coef_values = np.ravel(lin.named_steps['model'].coef_)
coef_names = lin.named_steps['prep'].get_feature_names_out()
usable = min(len(coef_values), len(coef_names))
coef = pd.Series(coef_values[:usable], index=coef_names[:usable]).sort_values(key=lambda s: s.abs(), ascending=False).head(10)
print('\nTop explanatory effects:')
print(coef.round(4).to_string())

print('\nDecision recommendations: trigger proactive staffing when rolling load and lagged load-per-resident signals rise vs house baseline.')


In [ ]:
import sys
from pathlib import Path
for candidate in [Path.cwd(), Path.cwd() / 'ml-pipelines', Path.cwd().parent]:
    if (candidate / 'trend_eval_helpers.py').exists():
        sys.path.insert(0, str(candidate))
        break
from trend_eval_helpers import print_calibration_bins, print_threshold_scan, bootstrap_linear_coefs, fairness_binary, fairness_regression_mae

print("\n=== Evaluation artifacts ===")
if y_cls.nunique() >= 2 and yte_clf.nunique() >= 2:
    proba = rf.predict_proba(Xte)[:, 1]
    print("\n--- Calibration bins (holdout) ---")
    print_calibration_bins(yte_clf.values, proba)
    print("\n--- Threshold scan (favor recall for safety ops) ---")
    print_threshold_scan(yte_clf.values, proba)
    fairness_binary(rf, Xte, yte_clf, m.loc[Xte.index], ['safehouse_id'], min_n=8)
else:
    print("Classifier calibration skipped (degenerate holdout or target).")
print("\n--- Bootstrap CI: explanatory linear (train rows) ---")
bootstrap_linear_coefs(lin, Xtr, ytr_reg, n_boot=150, top_k=8)
fairness_regression_mae(lin, Xte, yte_reg, m.loc[Xte.index], ['safehouse_id'], min_n=8)
